# Lesson 4: Building a Multi-Document Agent

## Setup

In [1]:
from helper import get_openai_api_key
OPENAI_API_KEY = get_openai_api_key()

In [2]:
import nest_asyncio
nest_asyncio.apply()

## 1. Setup an agent over 3 papers

**Note**: The pdf files are included with this lesson. To access these papers, go to the `File` menu and select`Open...`.

In [12]:
urls = [
   "https://s172-29-4-51p8888.lab-aws-production.deeplearning.ai/files/Lesson_4/33768_Hedging_on_the_Frontier_.pdf",
    "https://s172-29-4-51p8888.lab-aws-production.deeplearning.ai/files/Lesson_4/34048_Second_Order_Smooth_Plan.pdf",
    "https://s172-29-4-51p8888.lab-aws-production.deeplearning.ai/files/Lesson_4/34584_Foundations_of_Equivaria.pdf",
]

papers = [
    "33768_Hedging_on_the_Frontier_.pdf",
    "34048_Second_Order_Smooth_Plan.pdf",
    "34584_Foundations_of_Equivaria.pdf",
]

In [13]:
from utils import get_doc_tools
from pathlib import Path

paper_to_tools_dict = {}
for paper in papers:
    print(f"Getting tools for paper: {paper}")
    vector_tool, summary_tool = get_doc_tools(paper, Path(paper).stem)
    paper_to_tools_dict[paper] = [vector_tool, summary_tool]

Getting tools for paper: 33768_Hedging_on_the_Frontier_.pdf
Getting tools for paper: 34048_Second_Order_Smooth_Plan.pdf
Getting tools for paper: 34584_Foundations_of_Equivaria.pdf


In [20]:
initial_tools = [t for paper in papers for t in paper_to_tools_dict[paper]]

In [21]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-3.5-turbo")

In [22]:
len(initial_tools)

6

In [23]:
from llama_index.core.agent import FunctionCallingAgentWorker
from llama_index.core.agent import AgentRunner

agent_worker = FunctionCallingAgentWorker.from_tools(
    initial_tools, 
    llm=llm, 
    verbose=True
)
agent = AgentRunner(agent_worker)

In [24]:
response = agent.query(
    "Tell me about the Hedging on the frontier "
    "and then tell me about the evaluation results"
)

Added user message to memory: Tell me about the Hedging on the frontier and then tell me about the evaluation results
=== Calling Function ===
Calling function: summary_tool_33768_Hedging_on_the_Frontier_ with args: {"input": "Hedging on the frontier"}
=== Function Output ===
Hedging on the frontier involves a strategy in learning new tasks with limited data samples by leveraging weak monotonicity assumptions to select models that perform well across multiple benchmarks. This approach aims to identify potentially suitable models for the target task by considering a set of benchmarks and the relationship between them and the new task. The strategy involves selecting models that form a frontier of potentially effective choices, pruning dominated models, and being selective over the frontier models to avoid improper trade-offs. The effectiveness of hedging on the frontier is determined by the geometry induced by the weak monotonicity assumption, aiding in making informed model selections 

In [25]:
response = agent.query("Give me a summary of both Hedging on the frontier and Bellman Smoothing")
print(str(response))

Added user message to memory: Give me a summary of both Hedging on the frontier and Bellman Smoothing
=== Calling Function ===
Calling function: summary_tool_33768_Hedging_on_the_Frontier_ with args: {"input": "summary"}
=== Function Output ===
The research discussed in the provided context focuses on learning new tasks with limited data samples by leveraging benchmark evaluations and weak monotonicity assumptions. The study explores how models can be pruned based on benchmark performance and adapted to the available trade-offs by "hedging on the frontier." The concept of weak monotonicity is introduced, which suggests that models consistently better on a set of benchmarks are likely to perform better on the target task. The study delves into transfer learning and model selection aggregation paradigms under weak monotonicity, aiming to achieve efficient learning with limited data. The research also introduces the Pareto covering number as a complexity measure and provides sample comple